In [185]:
import scanpy as scp
import scipy.sparse as sprs
import pandas as pd
import warnings
import os

warnings.filterwarnings("ignore")

In [186]:
genes = pd.read_csv("../Preprocessing/results/5_counts/Genes.csv")
cells = pd.read_csv("../Preprocessing/results/5_counts/Cells.csv")
m = sprs.load_npz("../Preprocessing/results/5_counts/count_matrix_total.npz")

# Merging lanes

In [187]:
for j,i in zip(cells[cells["Lane"]==2].index,cells[cells["Lane"]==1].values):
    name = i[0]
    time = i[1]
    condition = i[2]

    cond1 = cells["Cell"] == name.replace("_1_","_2_")
    cond2 = cells["Time"] == time
    cond3 = cells["Lane"] == 1

    cell = cells[cond1.values*cond2.values*cond3.values]
    if cell.shape[0] == 1:
        m[j,:] += m[cell.index,:]

m = m[cells[cells["Lane"]==2].index]
cells = cells[cells["Lane"]==2].reset_index(drop=True)


In [188]:
adata = scp.AnnData(m,obs=cells,var=genes.set_index("gene_id"))

# Normalizing the tags

We can check that some tags have the same meaning but different nomenclature.

1. kitScal and kitscal in Condition
1. empty and EMPTY in BiologicalSample

We correct for this double taggings

In [189]:
for i in adata.obs.columns:
    print(i,": ", adata.obs[i].unique())

Cell :  ['HG3JFDRXY_2_IDT-DUI-NXT-193' 'HG3JFDRXY_2_IDT-DUI-NXT-194'
 'HG3JFDRXY_2_IDT-DUI-NXT-195' 'HG3JFDRXY_2_IDT-DUI-NXT-196'
 'HG3JFDRXY_2_IDT-DUI-NXT-197' 'HG3JFDRXY_2_IDT-DUI-NXT-198'
 'HG3JFDRXY_2_IDT-DUI-NXT-199' 'HG3JFDRXY_2_IDT-DUI-NXT-200'
 'HG3JFDRXY_2_IDT-DUI-NXT-201' 'HG3JFDRXY_2_IDT-DUI-NXT-202'
 'HG3JFDRXY_2_IDT-DUI-NXT-203' 'HG3JFDRXY_2_IDT-DUI-NXT-204'
 'HG3JFDRXY_2_IDT-DUI-NXT-205' 'HG3JFDRXY_2_IDT-DUI-NXT-206'
 'HG3JFDRXY_2_IDT-DUI-NXT-207' 'HG3JFDRXY_2_IDT-DUI-NXT-208'
 'HG3JFDRXY_2_IDT-DUI-NXT-209' 'HG3JFDRXY_2_IDT-DUI-NXT-210'
 'HG3JFDRXY_2_IDT-DUI-NXT-211' 'HG3JFDRXY_2_IDT-DUI-NXT-212'
 'HG3JFDRXY_2_IDT-DUI-NXT-213' 'HG3JFDRXY_2_IDT-DUI-NXT-214'
 'HG3JFDRXY_2_IDT-DUI-NXT-215' 'HG3JFDRXY_2_IDT-DUI-NXT-216'
 'HG3JFDRXY_2_IDT-DUI-NXT-217' 'HG3JFDRXY_2_IDT-DUI-NXT-218'
 'HG3JFDRXY_2_IDT-DUI-NXT-219' 'HG3JFDRXY_2_IDT-DUI-NXT-220'
 'HG3JFDRXY_2_IDT-DUI-NXT-221' 'HG3JFDRXY_2_IDT-DUI-NXT-222'
 'HG3JFDRXY_2_IDT-DUI-NXT-223' 'HG3JFDRXY_2_IDT-DUI-NXT-224'
 'HG3JFDRXY_2_ID

In [190]:
adata.obs.loc[adata.obs["Condition"]=="kitscal","Condition"] = "kitScal"
adata.obs.loc[adata.obs["BiologicalSample"]=="EMPTY","BiologicalSample"] = "empty"

# Adding Flow cytommetry measures to data

In [191]:
conditions = os.listdir("input/SMARTseq")
for condition in conditions:
    for file_num,file_name in enumerate(os.listdir("input/SMARTseq/"+condition)):

        data = pd.read_csv("input/SMARTseq/"+condition+"/"+file_name,skiprows=9)
        data.set_index("Well",inplace=True)

        table = pd.read_excel("input/Mapping.xlsx",sheet_name=condition).set_index("X1").unstack()

        m = {i[1]+str(i[0]):str(j).replace("_1_","_2_") for i,j in zip(table.index, table.values)}

        for i in data.index:
            for j in [i for i in data.columns if "All" in i]:
                adata.obs.loc[adata.obs["Cell"] == m[i],j] = data.loc[i,j]

# Save data

In [38]:
adata.write("results/Raw.h5ad")